# Heart Disease Classifier – Model Training & Experiment Tracking

**MLOps Assignment 01 – BITS Pilani (AIMLCZG523)**

This notebook covers:
1. Feature engineering & preprocessing pipeline
2. Model training (Logistic Regression, Random Forest, XGBoost)
3. Hyperparameter tuning with GridSearchCV
4. Cross-validation evaluation
5. MLflow experiment tracking
6. Model comparison & selection
7. Saving best model

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
import joblib

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay,
    classification_report,
)
from xgboost import XGBClassifier

from src.data_processing import (
    prepare_data, build_preprocessing_pipeline, get_feature_columns,
    NUMERICAL_FEATURES, CATEGORICAL_FEATURES,
)

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')
RANDOM_STATE = 7
print('Setup complete')

## 1 – Load & Prepare Data

In [ ]:
DATA_PATH = '../data/raw/heart_disease.csv'
X_train, X_test, y_train, y_test = prepare_data(DATA_PATH)
num_cols, cat_cols = get_feature_columns(X_train)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Numerical features: {num_cols}')
print(f'Categorical features: {cat_cols}')
print(f'Class distribution (train):\n{y_train.value_counts()}')

## 2 – MLflow Setup

In [ ]:
mlflow.set_tracking_uri('../mlruns')
mlflow.set_experiment('heart-disease-classifier-notebook')
print('MLflow tracking URI:', mlflow.get_tracking_uri())

## 3 – Helper: Train & Evaluate a Model

In [ ]:
CV = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

def train_and_evaluate(name, classifier, param_grid):
    with mlflow.start_run(run_name=name) as run:
        mlflow.set_tag('model', name)

        preprocessor = build_preprocessing_pipeline(num_cols, cat_cols)
        pipeline = Pipeline([('prep', preprocessor), ('clf', classifier)])
        prefixed = {f'clf__{k}': v for k, v in param_grid.items()}

        gs = GridSearchCV(pipeline, prefixed, cv=CV,
                          scoring='roc_auc', n_jobs=-1)
        gs.fit(X_train, y_train)

        best = gs.best_estimator_
        best_params = {k.replace('clf__', ''): v
                       for k, v in gs.best_params_.items()}
        mlflow.log_params(best_params)

        # Cross-validation
        cv_res = cross_validate(best, X_train, y_train, cv=CV,
                                scoring=['accuracy','f1','roc_auc'])
        for metric in ['accuracy','f1','roc_auc']:
            mlflow.log_metric(f'cv_{metric}', cv_res[f'test_{metric}'].mean())

        # Test metrics
        y_pred = best.predict(X_test)
        y_prob = best.predict_proba(X_test)[:,1]
        metrics = {
            'test_accuracy':  accuracy_score(y_test, y_pred),
            'test_precision': precision_score(y_test, y_pred),
            'test_recall':    recall_score(y_test, y_pred),
            'test_f1':        f1_score(y_test, y_pred),
            'test_roc_auc':   roc_auc_score(y_test, y_prob),
        }
        mlflow.log_metrics(metrics)

        # Plots
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        ConfusionMatrixDisplay.from_predictions(
            y_test, y_pred, display_labels=['No Disease','Disease'],
            cmap='Blues', ax=axes[0])
        axes[0].set_title(f'{name} – Confusion Matrix')
        RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
        axes[1].set_title(f'{name} – ROC Curve')
        plt.tight_layout()
        fig.savefig(f'../screenshots/{name}_eval.png', bbox_inches='tight')
        mlflow.log_artifact(f'../screenshots/{name}_eval.png')
        plt.show()

        mlflow.sklearn.log_model(best, artifact_path='model')
        print(f'\n{name} – Run ID: {run.info.run_id}')
        print(f'Best params: {best_params}')
        print(f'Metrics: {metrics}')
        print(classification_report(y_test, y_pred,
                                    target_names=["No Disease","Disease"]))

    return {'name': name, 'pipeline': best, 'metrics': metrics}

print('Helper defined')

## 4 – Model 1: Logistic Regression

In [ ]:
lr_result = train_and_evaluate(
    name='LogisticRegression',
    classifier=LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    param_grid={
        'C': [0.001, 0.01, 0.1, 1.0, 5.0],
        'solver': ['lbfgs', 'saga'],
        'penalty': ['l2'],
    },
)

## 5 – Model 2: Random Forest

In [ ]:
rf_result = train_and_evaluate(
    name='RandomForest',
    classifier=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid={
        'n_estimators': [150, 300],
        'max_depth': [None, 8, 15],
        'min_samples_split': [2, 4],
        'max_features': ['sqrt', 'log2'],
    },
)

## 6 – Model 3: XGBoost

In [ ]:
xgb_result = train_and_evaluate(
    name='XGBoost',
    classifier=XGBClassifier(eval_metric='logloss',
                              random_state=RANDOM_STATE, verbosity=0),
    param_grid={
        'n_estimators': [150, 300],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8, 1.0],
    },
)

## 6b – Model 4: Gradient Boosting

In [ ]:
results = [lr_result, rf_result, xgb_result, gb_result]
comparison = pd.DataFrame([
    {'Model': r['name'], **{k.replace('test_','').upper(): round(v,4)
                            for k,v in r['metrics'].items()}}
    for r in results
])
print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
metrics_to_plot = ['ACCURACY', 'PRECISION', 'RECALL', 'F1', 'ROC_AUC']
x = np.arange(len(metrics_to_plot))
width = 0.2
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
for i, r in enumerate(results):
    vals = [comparison.loc[i, m] for m in metrics_to_plot]
    ax.bar(x + i*width, vals, width, label=r['name'], color=colors[i])
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison – Test Set Metrics')
ax.legend()
plt.tight_layout()
plt.savefig('../screenshots/model_comparison.png', bbox_inches='tight')
plt.show()

## 7 – Model Comparison

In [ ]:
results = [lr_result, rf_result, xgb_result]
comparison = pd.DataFrame([
    {'Model': r['name'], **{k.replace('test_','').upper(): round(v,4)
                            for k,v in r['metrics'].items()}}
    for r in results
])
print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
metrics_to_plot = ['ACCURACY', 'PRECISION', 'RECALL', 'F1', 'ROC_AUC']
x = np.arange(len(metrics_to_plot))
width = 0.25
colors = ['#2196F3', '#4CAF50', '#FF9800']
for i, r in enumerate(results):
    vals = [comparison.loc[i, m] for m in metrics_to_plot]
    ax.bar(x + i*width, vals, width, label=r['name'], color=colors[i])
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison – Test Set Metrics')
ax.legend()
plt.tight_layout()
plt.savefig('../screenshots/model_comparison.png', bbox_inches='tight')
plt.show()

## 8 – Save Best Model

In [ ]:
import json

best = max(results, key=lambda r: r['metrics']['test_roc_auc'])
print(f"Best model: {best['name']}  ROC-AUC = {best['metrics']['test_roc_auc']:.4f}")

os.makedirs('../models', exist_ok=True)
joblib.dump(best['pipeline'], '../models/best_model.joblib')
print('Saved: ../models/best_model.joblib')

meta = {
    'model_name': best['name'],
    'metrics': {k: round(v,4) for k,v in best['metrics'].items()},
    'num_features': num_cols,
    'cat_features': cat_cols,
}
with open('../models/model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved: ../models/model_metadata.json')

## 9 – Quick Inference Test

Verify the saved pipeline produces correct predictions.

In [ ]:
from src.inference import load_model, predict

pipeline = load_model('../models/best_model.joblib')
sample = {
    'age': 63, 'sex': 1, 'cp': 3, 'trestbps': 145, 'chol': 233,
    'fbs': 1, 'restecg': 0, 'thalach': 150, 'exang': 0,
    'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 1,
}
result = predict(pipeline, sample)
print('Inference result:', result)